In [1]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
tf.get_logger().setLevel("ERROR")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/somemultibranch/')
sys.path.append('/kaggle/input/cmi-competition-code')

import pandas as pd
import data_utils
import os
import utils
from sklearn.pipeline import Pipeline
from scipy.stats import randint
from skopt.space import Categorical, Integer
from sklearn_genetic.space import Categorical as ECat, Integer as EInt
from sklearn.model_selection import GridSearchCV
from sklearn_genetic import GASearchCV
from sklearn.model_selection import RandomizedSearchCV
from skopt import BayesSearchCV
from sklearn.model_selection import GroupKFold
from skopt.space import Categorical, Integer, Real
from sklearn_genetic.space import Categorical as ECat, Integer as EInt, Continuous as EFloat
from scipy.stats import randint, uniform, loguniform
from sklearn.metrics import f1_score, make_scorer
import numpy as np

from sklearn.model_selection import KFold
import importlib
warnings.filterwarnings('ignore', module='deap')
import utils
from sklearn.metrics import accuracy_score, classification_report

2026-05-14 00:45:00.601069: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778719500.820473      31 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778719500.884713      31 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778719501.382630      31 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778719501.382673      31 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778719501.382676      31 computation_placer.cc:177] computation placer alr

In [2]:
data_folder = data_utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
n_splits = 3
cv = GroupKFold(n_splits=n_splits)
scoring_metric = 'f1_macro'
model_target = 'gesture_action'

scoring = None

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
acc_cols = ['acc_x', 'acc_y', 'acc_z']

pipe_name = "temporal_extractor"
classifier_name = 'multi_branch'

search_mode = "bayesian"  # grid, random, evolutionary, bayesian

pipe_name = 'temporal_extractor'
candidates = 15
generations = 2
tournament_size = 2
elitism = True
crossover_probability = 0.8
mutation_probability = 0.2

chosen_orientation = None

train_size = 0.4

experiment_notes = 'composite target and bigru testing thanks deepseek'

In [4]:
train_df = raw_train_df.set_index("row_id").copy(deep=True)

if train_size is None:
    rows = (
        (train_demo_df["adult_child"] == 1)
        & (train_demo_df["sex"] == 1)
        & (train_demo_df["handedness"] == 1)
    )

    ideal_subject_ids = (
        train_demo_df.loc[rows]
        .sort_values("elbow_to_wrist_cm", ascending=False)["subject"]
        .to_list()
    )

    train_sample_df = train_df.loc[
        train_df["subject"].isin(ideal_subject_ids) & (train_df["sequence_type"] == "Target")
    ].copy()  # ← ALREADY HAVE .copy() HERE - GOOD!
    
    test_sample_df = None

elif train_size == 0:
    some_sequences = train_df["sequence_id"].unique()[10:20]

    train_sample_df = train_df.loc[
        train_df["sequence_id"].isin(some_sequences)
    ].copy()  # ← ADD .copy() HERE
    
    test_sample_df = None

else:
    target_df = train_df[train_df["sequence_type"] == "Target"].copy()
    
    train_sample_df, test_sample_df = data_utils.sample_balanced_split(
        target_df,
        train_pct=train_size,
        test_pct=0.2,
    )
    # Make sure both are copies
    train_sample_df = train_sample_df.copy()
    if test_sample_df is not None:
        test_sample_df = test_sample_df.copy()

# NOW it's safe to modify - use .loc to be explicit
train_sample_df.loc[:, "gesture_position"] = train_sample_df["gesture"].str.split(" - ").str[0]
train_sample_df.loc[:, "gesture_action"] = train_sample_df["gesture"].str.split(" - ").str[-1]
train_sample_df.loc[:, "composite_target"] = (
    train_sample_df["orientation"].astype(str) + "_" +
    train_sample_df["gesture_action"] + "_" +
    train_sample_df["phase"]
)
train_sample_df.loc[:, "phase_target"] = train_sample_df["phase"]

if test_sample_df is not None and not test_sample_df.empty:
    test_sample_df.loc[:, "gesture_position"] = test_sample_df["gesture"].str.split(" - ").str[0]
    test_sample_df.loc[:, "gesture_action"] = test_sample_df["gesture"].str.split(" - ").str[-1]
    test_sample_df.loc[:, "composite_target"] = (
        test_sample_df["orientation"].astype(str) + "_" +
        test_sample_df["gesture_action"] + "_" +
        test_sample_df["phase"]
    )
    test_sample_df.loc[:, "phase_target"] = test_sample_df["phase"]

# Apply orientation filter to train only
if chosen_orientation is not None and train_sample_df is not None:
    train_sample_df = train_sample_df.loc[
        train_sample_df["orientation"].isin(chosen_orientation)
    ].copy()

if train_sample_df is not None:
    print(f"Train sequences: {train_sample_df['sequence_id'].nunique()}")
if test_sample_df is not None:
    print(f"Test sequences: {test_sample_df['sequence_id'].nunique()}")

Train: 1944 seqs | 38.0%
Test:  648 seqs  | 12.7%
Train sequences: 1944
Test sequences: 648


In [5]:
if search_mode == "bayesian":
    param_space = {
        # --- Preprocessing ---
        f"{pipe_name}__acc_mode": Categorical(["raw", "displacement"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__sampling_rate": Categorical([10, 20, 100, 50]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear"]),
        f"{pipe_name}__use_highpass_fallback": Categorical([True]),
        f"{pipe_name}__window_size": Integer(3, 21),
        f"{pipe_name}__smooth_alpha": Categorical([None, 0.2, 0.5, 0.8]),
        f"{pipe_name}__standardize": Categorical([ "mean_std"]),
        f"{pipe_name}__include_mask": Categorical([True]),

        f"{pipe_name}__rotation_mode": Categorical(["quaternion", "euler", "rot6d"]),
        f"{pipe_name}__fix_quaternion_sign": Categorical([True]),

        f"{pipe_name}__tof_mode": Categorical( ["pooled_diff", "sensor_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["far_255"]),
        f"{pipe_name}__thm_mode": Categorical(["centered_diff"]),

        # --- Classifier ---
        f"{classifier_name}__maxlen": Integer(10, 200),
        f"{classifier_name}__padding_value": Categorical([-999.0]),

        # fusion
        f"{classifier_name}__fusion_mode": Categorical(["bigru"]),
        f"{classifier_name}__attention_heads": Integer(2, 8),
        f"{classifier_name}__gru_units": Integer(64, 256),

        # per-branch filters (common patterns)
        f"{classifier_name}__branch_filters": [
            {"acc": "64", "rot": "32", "tof": "16", "thm": "8"},
            {"acc": "64-128", "rot": "32-64", "tof": "32", "thm": "16"},
            {"acc": "64-128-128", "rot": "64-64", "tof": "64", "thm": "16"},
            {"acc": "128", "rot": "64", "tof": "32", "thm": "16"},
        ],

        f"{classifier_name}__branch_kernel_sizes": [
            {"acc": "3", "rot": "3", "tof": "3", "thm": "3"},
            #{"acc": "3-3", "rot": "3-3", "tof": "3", "thm": "3"},
            #{"acc": "5-3", "rot": "5-3", "tof": "5", "thm": "3"},
        ],

        f"{classifier_name}__branch_pool_sizes": [
            {"acc": "none", "rot": "none", "tof": "none", "thm": "none"},
            #{"acc": "none-none", "rot": "none-none", "tof": "none", "thm": "none"},
            #{"acc": "2-none", "rot": "none-none", "tof": "none", "thm": "none"},
        ],

        # common
        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__spatial_dropout": Real(0.0, 0.3),
        f"{classifier_name}__dense_units": Categorical(["none", "32", "64", "128", "64-32", "128-64"]),
        f"{classifier_name}__dropout": Real(0.0, 0.5),
        f"{classifier_name}__learning_rate": Real(1e-4, 2e-3, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([16, 32, 64]),
        f"{classifier_name}__epochs": Categorical([60, 80, 120]),
        f"{classifier_name}__patience": Categorical([8, 12, 20]),
    }

elif search_mode == "evolutionary":
    param_space = {
        # --- Preprocessing ---
        f"{pipe_name}__acc_mode": ECat(["raw", "smoothed", "velocity", "displacement", "jerk"]),
        f"{pipe_name}__linear_acc_mode": ECat([None, "baseline"]),
        f"{pipe_name}__use_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__use_linear_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__sampling_rate": ECat([20, 25, 50]),
        f"{pipe_name}__compute_dt": ECat([True]),
        f"{pipe_name}__clip_value": ECat([None, 20.0, 50.0, 100.0]),
        f"{pipe_name}__interp_mode": ECat([None, "linear"]),
        f"{pipe_name}__use_highpass_fallback": ECat([True]),
        f"{pipe_name}__window_size": EInt(3, 21),
        f"{pipe_name}__smooth_alpha": ECat([None, 0.2, 0.5, 0.8]),
        f"{pipe_name}__standardize": ECat([None, "mean_std"]),
        f"{pipe_name}__include_mask": ECat([False]),

        f"{pipe_name}__rotation_mode": ECat([None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"]),
        f"{pipe_name}__fix_quaternion_sign": ECat([True, False]),

        f"{pipe_name}__tof_mode": ECat([None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": ECat(["nan_interpolate", "far_255", "far_500"]),
        f"{pipe_name}__thm_mode": ECat([None, "raw", "diff", "centered", "centered_diff"]),

        # --- Classifier ---
        f"{classifier_name}__maxlen": EInt(16, 160),
        f"{classifier_name}__padding_value": ECat([-999.0]),

        # fusion
        f"{classifier_name}__fusion_mode": ECat(["attention", "bigru", "none"]),
        f"{classifier_name}__attention_heads": EInt(2, 8),
        f"{classifier_name}__gru_units": EInt(64, 256),

        # per-branch filters
        f"{classifier_name}__branch_filters": ECat([
            {"acc":"64","rot":"32","tof":"16","thm":"8"},
            {"acc":"64-128","rot":"32-64","tof":"32","thm":"16"},
            {"acc":"64-128-128","rot":"64-64","tof":"64","thm":"16"},
            {"acc":"128","rot":"64","tof":"32","thm":"16"},
        ]),
        f"{classifier_name}__branch_kernel_sizes": ECat([
            {"acc":"3","rot":"3","tof":"3","thm":"3"},
            {"acc":"3-3","rot":"3-3","tof":"3","thm":"3"},
            {"acc":"5-3","rot":"5-3","tof":"5","thm":"3"},
        ]),
        f"{classifier_name}__branch_pool_sizes": ECat([
            {"acc":"none","rot":"none","tof":"none","thm":"none"},
            {"acc":"none-none","rot":"none-none","tof":"none","thm":"none"},
            {"acc":"2-none","rot":"none-none","tof":"none","thm":"none"},
        ]),

        # common
        f"{classifier_name}__use_batch_norm": ECat([True, False]),
        f"{classifier_name}__spatial_dropout": EFloat(0.0, 0.3),
        f"{classifier_name}__dense_units": ECat(["none", "32", "64", "128", "64-32", "128-64"]),
        f"{classifier_name}__dropout": EFloat(0.0, 0.5),
        f"{classifier_name}__learning_rate": EFloat(1e-4, 2e-3),
        f"{classifier_name}__batch_size": ECat([16, 32, 64]),
        f"{classifier_name}__epochs": ECat([60, 80, 120]),
        f"{classifier_name}__patience": ECat([8, 12, 20]),
    }

elif search_mode == "random":
    param_space = {
        # --- Preprocessing ---
        f"{pipe_name}__acc_mode": ["raw", "smoothed", "velocity", "displacement", "jerk"],
        f"{pipe_name}__linear_acc_mode": [None, "baseline"],
        f"{pipe_name}__use_acc_magnitude": [False, True],
        f"{pipe_name}__use_linear_acc_magnitude": [False, True],
        f"{pipe_name}__sampling_rate": [20, 25, 50],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [None, 20.0, 50.0, 100.0],
        f"{pipe_name}__interp_mode": [None, "linear"],
        f"{pipe_name}__use_highpass_fallback": [True],
        f"{pipe_name}__window_size": randint(3, 22),
        f"{pipe_name}__smooth_alpha": [None, 0.2, 0.5, 0.8],
        f"{pipe_name}__standardize": [None, "mean_std"],
        f"{pipe_name}__include_mask": [False],

        f"{pipe_name}__rotation_mode": [None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"],
        f"{pipe_name}__fix_quaternion_sign": [False, True],

        f"{pipe_name}__tof_mode": [None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"],
        f"{pipe_name}__tof_fill_mode": ["nan_interpolate", "far_255", "far_500"],
        f"{pipe_name}__thm_mode": [None, "raw", "diff", "centered", "centered_diff"],

        # --- Classifier ---
        f"{classifier_name}__maxlen": randint(16, 161),
        f"{classifier_name}__padding_value": [-999.0],

        # fusion
        f"{classifier_name}__fusion_mode": ["attention", "bigru", "none"],
        f"{classifier_name}__attention_heads": randint(2, 9),
        f"{classifier_name}__gru_units": randint(64, 257),

        # per-branch filters
        f"{classifier_name}__branch_filters": [
            {"acc":"64","rot":"32","tof":"16","thm":"8"},
            {"acc":"64-128","rot":"32-64","tof":"32","thm":"16"},
            {"acc":"64-128-128","rot":"64-64","tof":"64","thm":"16"},
            {"acc":"128","rot":"64","tof":"32","thm":"16"},
        ],
        f"{classifier_name}__branch_kernel_sizes": [
            {"acc":"3","rot":"3","tof":"3","thm":"3"},
            {"acc":"3-3","rot":"3-3","tof":"3","thm":"3"},
            {"acc":"5-3","rot":"5-3","tof":"5","thm":"3"},
        ],
        f"{classifier_name}__branch_pool_sizes": [
            {"acc":"none","rot":"none","tof":"none","thm":"none"},
            {"acc":"none-none","rot":"none-none","tof":"none","thm":"none"},
            {"acc":"2-none","rot":"none-none","tof":"none","thm":"none"},
        ],

        # common
        f"{classifier_name}__use_batch_norm": [True, False],
        f"{classifier_name}__spatial_dropout": uniform(0.0, 0.3),
        f"{classifier_name}__dense_units": ["none", "32", "64", "128", "64-32", "128-64"],
        f"{classifier_name}__dropout": uniform(0.0, 0.5),
        f"{classifier_name}__learning_rate": loguniform(1e-4, 2e-3),
        f"{classifier_name}__batch_size": [16, 32, 64],
        f"{classifier_name}__epochs": [60, 80, 120],
        f"{classifier_name}__patience": [8, 12, 20],
    }

elif search_mode == "grid":
    param_space = {
        # --- Preprocessing (locked to best from previous runs) ---
        f"{pipe_name}__acc_mode": ["raw", "smoothed", "velocity", "displacement", "jerk"],
        f"{pipe_name}__linear_acc_mode": ["baseline"],
        f"{pipe_name}__use_acc_magnitude": [True],
        f"{pipe_name}__use_linear_acc_magnitude": [True],
        f"{pipe_name}__sampling_rate": [100],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [50.0],
        f"{pipe_name}__interp_mode": ["linear"],
        f"{pipe_name}__use_highpass_fallback": [True],
        f"{pipe_name}__window_size": [50],
        f"{pipe_name}__smooth_alpha": [0.8],
        f"{pipe_name}__standardize": ["mean_std"],
        f"{pipe_name}__include_mask": [False],

        f"{pipe_name}__rotation_mode": ["rot6d"],
        f"{pipe_name}__fix_quaternion_sign": [True],

        f"{pipe_name}__tof_mode": ["pooled_diff"],
        f"{pipe_name}__tof_fill_mode": ["far_255"],
        f"{pipe_name}__thm_mode": ["centered_diff"],

        # --- Classifier (vary architecture only) ---
        f"{classifier_name}__maxlen": [120],
        f"{classifier_name}__padding_value": [-999.0],

        # try each fusion mode
        f"{classifier_name}__fusion_mode": ["attention"],
        f"{classifier_name}__attention_heads": [4],
        f"{classifier_name}__gru_units": [128],

        # try different branch depths
        f"{classifier_name}__branch_filters": [
            {"acc":"64","rot":"32","tof":"16","thm":"8"},
            # {"acc":"64-128","rot":"32-64","tof":"32","thm":"16"},
            # {"acc":"64-128-128","rot":"64-64","tof":"64","thm":"16"},
        ],
        f"{classifier_name}__branch_kernel_sizes": [
            {"acc":"3","rot":"3","tof":"3","thm":"3"},
            # {"acc":"3-3","rot":"3-3","tof":"3","thm":"3"},
        ],
        f"{classifier_name}__branch_pool_sizes": [
            {"acc":"none","rot":"none","tof":"none","thm":"none"},
        ],

        # common
        f"{classifier_name}__use_batch_norm": [True],
        f"{classifier_name}__spatial_dropout": [0.1],
        f"{classifier_name}__dense_units": ["64"],
        f"{classifier_name}__dropout": [0.3],
        f"{classifier_name}__learning_rate": [5e-4],
        f"{classifier_name}__batch_size": [32],
        f"{classifier_name}__epochs": [80],
        f"{classifier_name}__patience": [12],
    }

param_space = utils.prepare_multibranch_param_space(
    param_space,
    search_mode,
    Categorical=Categorical,
    ECat=ECat,
)

In [6]:
importlib.reload(utils)         

pipeline = Pipeline([
    (pipe_name, utils.SequenceExtractor()),   # unchanged
    (classifier_name, utils.KerasMultiBranchClassifier(
        target=model_target,
        fusion_mode="attention",    # or "bigru"
        batch_size=32,
        epochs=80,
        patience=12,
    )),
])

if search_mode == "bayesian":
    search_obj = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "random":
    search_obj = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score = np.nan
    )

elif search_mode == "evolutionary":
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    search_obj = GASearchCV(
        estimator=pipeline,
        cv=cv,
        scoring=scoring,
        param_grid=param_space,
        population_size=candidates,
        generations=generations,
        tournament_size=tournament_size,
        elitism=elitism,
        crossover_probability=crossover_probability,
        mutation_probability=mutation_probability,
        criteria="max",
        n_jobs=1,
        verbose=True,
        keep_top_k=5,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "grid":
    search_obj = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=4,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

else:
    raise ValueError("search_mode must be one of: 'bayesian', 'random', 'evolutionary', 'grid'")

In [7]:
y = train_sample_df[['sequence_id', model_target]]
groups = train_sample_df['sequence_id']

print(f"--- {search_mode} Search ---")
if search_mode == 'evolutionary':
    search_obj.fit(train_sample_df, y)
else:
    search_obj.fit(train_sample_df, y, groups=groups)

--- bayesian Search ---
Fitting 3 folds for each of 1 candidates, totalling 3 fits


I0000 00:00:1778719709.204286      31 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778719709.210296      31 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778719717.083854      74 cuda_dnn.cc:529] Loaded cuDNN version 91002


[CV 1/3] END multi_branch__attention_heads=4, multi_branch__batch_size=64, multi_branch__branch_filters={"acc": "128", "rot": "64", "thm": "16", "tof": "32"}, multi_branch__branch_kernel_sizes={"acc": "3", "rot": "3", "thm": "3", "tof": "3"}, multi_branch__branch_pool_sizes={"acc": "none", "rot": "none", "thm": "none", "tof": "none"}, multi_branch__dense_units=64, multi_branch__dropout=0.175465667449572, multi_branch__epochs=120, multi_branch__fusion_mode=bigru, multi_branch__gru_units=188, multi_branch__learning_rate=0.0005194210966541066, multi_branch__maxlen=34, multi_branch__padding_value=-999.0, multi_branch__patience=12, multi_branch__spatial_dropout=0.023367368741176966, multi_branch__use_batch_norm=True, temporal_extractor__acc_mode=displacement, temporal_extractor__clip_value=50.0, temporal_extractor__compute_dt=True, temporal_extractor__fix_quaternion_sign=True, temporal_extractor__include_mask=True, temporal_extractor__interp_mode=linear, temporal_extractor__linear_acc_mode=

In [8]:
# --- 1. Model Prediction & Evaluation ---
best_model = search_obj.best_estimator_
X_test = test_sample_df.copy()

# Get unique ground truth labels per sequence
y_true_seq = (test_sample_df[['sequence_id', model_target]]
              .drop_duplicates('sequence_id')
              .reset_index(drop=True))

y_pred_seq = best_model.predict(X_test)

# Calculate Accuracy
test_accuracy = accuracy_score(y_true_seq[model_target], y_pred_seq)

print(f"--- Final Test Results ---")
print(f"Best CV Score: {search_obj.best_score_:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true_seq[model_target], y_pred_seq))

# --- 2. Cleanly Append Test Results to CV Results ---
if hasattr(search_obj, 'cv_results_'):
    # Convert search results to DataFrame
    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    cv_results_df['search_mode'] = search_mode
    cv_results_df['target'] = model_target
    
    # Create a "Final Test" row matching the CV columns
    # We use 'params' to label it and put the accuracy in 'mean_test_score'
    test_result_row = pd.DataFrame({
        'params': ['FINAL_HOLD_OUT_TEST'],
        'mean_test_score': [test_accuracy],
        'std_test_score': [0],
        'rank_test_score': [0],
        'exp_notes': [experiment_notes],
    })
    
    # Concat results - holes in the table (like split scores) fill with NaN
    final_report_df = pd.concat([cv_results_df, test_result_row], ignore_index=True)
    
    # Save the consolidated report
    file_path = f"{model_run_folder_name}{search_mode}_{classifier_name}_results.csv"
    final_report_df.to_csv(file_path, index=False)
    
    print(f"Results consolidated and saved to: {file_path}")

--- Final Test Results ---
Best CV Score: 0.5787
Test Accuracy: 0.6157

Classification Report:
                precision    recall  f1-score   support

   pinch skin       0.56      0.60      0.58       162
    pull hair       0.63      0.79      0.70       243
pull hairline       0.64      0.51      0.57        81
      scratch       0.63      0.43      0.51       162

     accuracy                           0.62       648
    macro avg       0.62      0.58      0.59       648
 weighted avg       0.62      0.62      0.61       648

Results consolidated and saved to: model_runs/bayesian_multi_branch_results.csv
